In [47]:
import numpy as np
import torch
import matplotlib as plt
import seaborn
from monai.transforms import LoadImaged, Compose
import json
import os
import re
from collections import Counter
import pandas as pd

In [48]:
from omegaconf import OmegaConf
from scripts.data_loader import get_loaders

In [50]:
from omegaconf import OmegaConf

In [51]:
from omegaconf import OmegaConf

cfg = OmegaConf.load("configs/default_config.yaml")

In [52]:
cfg = cfg['komondor']
cfg['batch_size'] = 1

In [53]:
from scripts.data_loader import ExtractSlabsAroundZ
import json

train_json = json.load(open('data/train.json'))
sample_dict = train_json[-10]

In [54]:
from monai.transforms import Compose, LoadImaged, EnsureChannelFirstd, ScaleIntensityRanged, Spacingd, ResizeWithPadOrCropd, EnsureTyped

In [55]:
import json
from monai.transforms import Compose, LoadImaged, EnsureTyped

val_list = json.load(open(cfg["val_list"], "r"))

transform = Compose([
    LoadImaged(keys=["ct"]),
    EnsureTyped(keys=["ct", "condition"])
])

In [70]:
import json
from monai.transforms import Compose, LoadImaged, EnsureTyped


train_list = json.load(open(cfg["train_list"], "r"))


transform = Compose([
    LoadImaged(keys=["ct", "gt_dose"]),
    EnsureTyped(
        keys=[
            "ct",
            "gt_dose",
            "condition",
            "ray_source",
            "ray_target",
        ]
    )
])


sample = transform(train_list[1])
sample

{'ct': metatensor([[[-1024., -1024., -1024.,  ..., -1024., -1024., -1024.],
          [-1024., -1024., -1024.,  ..., -1024., -1024., -1024.],
          [-1024., -1024., -1024.,  ..., -1024., -1024., -1024.],
          ...,
          [-1024., -1024., -1024.,  ..., -1024., -1024., -1024.],
          [-1024., -1024., -1024.,  ..., -1024., -1024., -1024.],
          [-1024., -1024., -1024.,  ..., -1024., -1024., -1024.]],
 
         [[-1024., -1024., -1024.,  ..., -1024., -1024., -1024.],
          [-1024., -1024., -1024.,  ..., -1024., -1024., -1024.],
          [-1024., -1024., -1024.,  ..., -1024., -1024., -1024.],
          ...,
          [-1024., -1024., -1024.,  ..., -1024., -1024., -1024.],
          [-1024., -1024., -1024.,  ..., -1024., -1024., -1024.],
          [-1024., -1024., -1024.,  ..., -1024., -1024., -1024.]],
 
         [[-1024., -1024., -1024.,  ..., -1024., -1024., -1024.],
          [-1024., -1024., -1024.,  ..., -1024., -1024., -1024.],
          [-1024., -1024., -10

In [75]:
import json
from monai.transforms import Compose, LoadImaged, EnsureTyped

val_list = json.load(open(cfg["val_list"], "r"))

transform = Compose([
    LoadImaged(keys=["ct"]),
    EnsureTyped(keys=["ct", "condition"])
])

In [76]:
sample = transform(train_list[1])

print(sample["gt_dose"].shape)
print(sample["condition"])

AttributeError: 'str' object has no attribute 'shape'

In [77]:
def get_beam_axis_2d(sample, flip_lps_to_ras=True):
    source = np.asarray(
        sample["ray_source"],
        dtype=np.float64,
    ).copy()

    target = np.asarray(
        sample["ray_target"],
        dtype=np.float64,
    ).copy()

    if flip_lps_to_ras:
        source[0] *= -1
        source[1] *= -1

        target[0] *= -1
        target[1] *= -1

    beam_dir_world = target - source
    beam_dir_world /= np.linalg.norm(beam_dir_world)

    affine = sample["gt_dose"].affine.detach().cpu().numpy()

    # world direction -> voxel direction
    voxel_dir = (
        np.linalg.inv(affine[:3, :3])
        @ beam_dir_world
    )

    # a [:,:,z] slice két tengelye
    axis_2d = voxel_dir[:2]

    norm = np.linalg.norm(axis_2d)

    if norm < 1e-8:
        return None

    axis_2d /= norm

    return axis_2d

In [78]:
def measure_beam_length_2d(
    sample,
    z_index=16,
    threshold=0.01,
):
    dose = (
        sample["gt_dose"][0, :, :, z_index]
        .detach()
        .cpu()
        .numpy()
    )

    if dose.max() <= 0:
        return None

    mask = (
        dose >= threshold * dose.max()
    )

    points = np.argwhere(mask).astype(np.float64)

    if len(points) < 2:
        return None

    axis = get_beam_axis_2d(sample)

    if axis is None:
        return None

    # minden pixel vetülete a beam tengelyére
    projection = points @ axis

    s_start = projection.min()
    s_end = projection.max()

    length_voxel_units = s_end - s_start

    # Egy egységnyi elmozdulás ezen a 2D voxelirányon
    # fizikailag hány mm?
    affine = sample["gt_dose"].affine.detach().cpu().numpy()

    delta_voxel = np.array([
        axis[0],
        axis[1],
        0.0,
    ])

    delta_world = (
        affine[:3, :3]
        @ delta_voxel
    )

    mm_per_unit = np.linalg.norm(
        delta_world
    )

    length_mm = (
        length_voxel_units
        * mm_per_unit
    )

    return float(length_mm)

In [79]:
length = measure_beam_length_2d(
    sample,
    z_index=16,
    threshold=0.01,
)

print(
    "Energy:",
    float(sample["condition"][2]),
    "MeV"
)

print(
    "True beam length:",
    length,
    "mm"
)

TypeError: string indices must be integers

In [62]:
def plot_measured_beam(sample, z_index=16, threshold=0.01):
    dose = (
        sample["gt_dose"][0, :, :, z_index]
        .detach()
        .cpu()
        .numpy()
    )

    mask = (
        dose >= threshold * dose.max()
    )

    points = np.argwhere(mask).astype(np.float64)

    axis = get_beam_axis_2d(sample)

    projection = points @ axis

    s0 = projection.min()
    s1 = projection.max()

    # Dose-weighted centroid
    weights = dose[mask]

    center = np.sum(
        points * weights[:, None],
        axis=0,
    ) / weights.sum()

    center_projection = center @ axis

    p_start = (
        center
        + (s0 - center_projection) * axis
    )

    p_end = (
        center
        + (s1 - center_projection) * axis
    )

    length = measure_beam_length_2d(
        sample,
        z_index=z_index,
        threshold=threshold,
    )

    plt.figure(figsize=(8, 8))

    plt.imshow(dose)

    # matplotlib: x = második index, y = első index
    plt.plot(
        [p_start[1], p_end[1]],
        [p_start[0], p_end[0]],
        linewidth=2,
        label=f"Measured beam: {length:.1f} mm",
    )

    plt.scatter(
        p_start[1],
        p_start[0],
        s=60,
        label="Start",
    )

    plt.scatter(
        p_end[1],
        p_end[0],
        s=60,
        label="End",
    )

    plt.legend()
    plt.tight_layout()
    plt.show()

In [81]:
import matplotlib as plt

plot_measured_beam(
    sample,
    z_index=16,
    threshold=0.01,
)

TypeError: string indices must be integers